<a href="https://colab.research.google.com/github/syoon7079/tjwn/blob/main/12_sensitivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#11주차 노트북 준비

In [1]:
# =============================================================================
# [12주차 민감도 분석 노트북 시작부] 11주차 전처리 + 완전 사례 + 모형 2 재현
# =============================================================================
!pip install pyreadstat -q

import os
import numpy as np
import pandas as pd
import pyreadstat
import statsmodels.formula.api as smf
from IPython.display import display, HTML
from google.colab import drive

drive.mount('/content/drive')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

DATA_DIR = '/content/drive/MyDrive/hdac_data'
SAV_PATH = f'{DATA_DIR}/HN24_ALL.sav'
AGE_MIN = 19
WEIGHT = 'wt_tot'

# 1) 변수 로드 (신체활동 pa_aerobic 포함)
NEEDED = ['year', 'l_out_fq', 'he_bmi', 'bd1_11', 'region', 'sex', 'age', 'educ', 'incm', 'bs3_1',
          'kstrata', 'psu', WEIGHT, 'pa_aerobic']
_, m0 = pyreadstat.read_sav(SAV_PATH, metadataonly=True)
low2real = {c.lower(): c for c in m0.column_names}
df, meta = pyreadstat.read_sav(SAV_PATH, usecols=[low2real[c] for c in NEEDED])
df.columns = df.columns.str.lower()

def num(s): return pd.to_numeric(s, errors='coerce')

# 2) 노출: 외식 빈도 3범주 (기준집단 low)
EXP_VALID = [1, 2, 3, 4, 5, 6, 7]
GRP_MAP   = {5: 'low', 6: 'low', 7: 'low', 4: 'mid', 1: 'high', 2: 'high', 3: 'high'}
df['exp_raw']   = num(df['l_out_fq'])
df['exp_valid'] = df['exp_raw'].isin(EXP_VALID)
df['grp']       = df['exp_raw'].where(df['exp_valid']).map(GRP_MAP)
df['grp']       = pd.Categorical(df['grp'], categories=['low', 'mid', 'high'])

# 3) 결과, 연령, 가중치
df['bmi']   = num(df['he_bmi'])
df['age_n'] = num(df['age'])
df['w']     = num(df[WEIGHT])

# 4) 공변량 (11주차와 동일)
df['c_sex']   = num(df['sex']).map({1: '남성', 2: '여성'})
df['c_educ']  = num(df['educ']).mask(num(df['educ']).isin([88, 99])).map({1: '초졸 이하', 2: '초졸 이하', 3: '초졸 이하', 4: '중졸', 5: '고졸', 6: '대졸 이상', 7: '대졸 이상', 8: '대졸 이상'})
df['c_incm']  = num(df['incm']).mask(num(df['incm']).isin([8, 9, 99])).map({1: '하', 2: '중하', 3: '중상', 4: '상'})
df['c_bd1_11']= num(df['bd1_11']).mask(num(df['bd1_11']).isin([9, 99])).map({1: '전혀 마시지 않음', 2: '월 1회 미만', 3: '월 1회 정도', 4: '월 2-4회', 5: '주 2-3회', 6: '주 4회 이상', 8: '비해당(코드 8)'})
df['c_bs3_1'] = num(df['bs3_1']).mask(num(df['bs3_1']).isin([9, 99])).map({1: '매일 흡연', 2: '가끔 흡연', 3: '과거 흡연(현재 비흡연)', 8: '비해당(코드 8)'})

# 5) 신체활동 (S3용)
df['c_pa'] = num(df['pa_aerobic']).mask(num(df['pa_aerobic']).isin([8, 9, 99]))

# 6) 설계변수 및 analytic
psu_str = df['psu'].astype(str).str.strip()
df['psu_ok']    = df['psu'].notna() & ~psu_str.isin(['', 'nan', 'None'])
df['design_ok'] = df['w'].notna() & (df['w'] > 0) & df['kstrata'].notna() & df['psu_ok']
df['analytic']  = (df['age_n'] >= AGE_MIN) & df['exp_valid'] & df['bmi'].notna() & df['design_ok']

# 7) 완전 사례
COVS = ['c_sex', 'region', 'c_educ', 'c_incm', 'c_bd1_11', 'c_bs3_1', 'age_n']
valid_mask = df['analytic'] & df['grp'].notna() & df[COVS].notna().all(axis=1)
sub_df = df[valid_mask].copy()
sub_df['grp'] = pd.Categorical(sub_df['grp'], categories=['low', 'mid', 'high'])
print(f"최종 완전 사례 분석 인원 (N): {len(sub_df):,}명 (11주차와 동일)")

# 8) 모형 함수
def fit_wls(formula, data):
    return smf.wls(formula, data=data, weights=data['w']).fit(
        cov_type='cluster', cov_kwds={'groups': data['psu']})

EXPO = "C(grp, Treatment(reference='low'))"
COV_TERMS = "C(c_sex) + C(region) + C(c_educ) + C(c_incm) + C(c_bd1_11) + C(c_bs3_1) + age_n"

def term(res, level):
    hits = [n for n in res.params.index if n.startswith('C(grp') and n.endswith(f'[T.{level}]')]
    assert len(hits) == 1, hits
    return hits[0]

# 9) 모형 2 재현 확인 표
res_m2 = fit_wls(f"bmi ~ {EXPO} + {COV_TERMS}", sub_df)

def format_result(res, level):
    t = term(res, level)
    b, se, p = res.params[t], res.bse[t], res.pvalues[t]
    lo, hi = res.conf_int().loc[t]
    ps = f"{p:.3f}" if p >= 0.001 else "< 0.001"
    return f"β = {b:.3f} (SE = {se:.3f})<br>95% CI: [{lo:.3f}, {hi:.3f}]<br>p-value = {ps}"

check_df = pd.DataFrame([
    {
        "구분": "12주차 재현 결과 (모형 2)",
        "분석 인원 (N)": f"{len(sub_df):,}명",
        "저빈도 외식군 (주 1-2회 이하)": "Ref. (-)",
        "중빈도 외식군 (주 3-4회)": format_result(res_m2, 'mid'),
        "고빈도 외식군 (주 5회 이상)": format_result(res_m2, 'high')
    }
])

print("=== [12주차 노트북: 모형 2 재현 확인 표] ===")
display(HTML(check_df.to_html(index=False, escape=False)))

# 11주차 결과(고빈도 0.474, 중빈도 0.194)와 다르면 오류로 알려줌 (같으면 아무것도 안 나옴)
assert abs(res_m2.params[term(res_m2, 'high')] - 0.474) < 0.001
assert abs(res_m2.params[term(res_m2, 'mid')] - 0.194) < 0.001

# 10) 신체활동 값 구성 확인 (S3 전에 눈으로 확인)
print("\n[pa_aerobic 원값 분포: analytic 내]")
print(df.loc[df['analytic'], 'pa_aerobic'].value_counts(dropna=False))
print("c_pa 결측 수 (analytic 내):", int(df.loc[df['analytic'], 'c_pa'].isna().sum()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
최종 완전 사례 분석 인원 (N): 5,602명 (11주차와 동일)
=== [12주차 노트북: 모형 2 재현 확인 표] ===


구분,분석 인원 (N),저빈도 외식군 (주 1-2회 이하),중빈도 외식군 (주 3-4회),고빈도 외식군 (주 5회 이상)
12주차 재현 결과 (모형 2),"5,602명",Ref. (-),"β = 0.194 (SE = 0.197)95% CI: [-0.193, 0.581]p-value = 0.326","β = 0.474 (SE = 0.153)95% CI: [0.175, 0.774]p-value = 0.002"



[pa_aerobic 원값 분포: analytic 내]
pa_aerobic
0.0    2979
1.0    2377
NaN     409
Name: count, dtype: int64
c_pa 결측 수 (analytic 내): 409


#D2 Table3

In [2]:
# ==============================================================================
# [D-2] Table 3: 주 분석 + 민감도 분석 S3, S4 (표로 출력)
# 전제: 셀 1(시작부)이 실행되어 sub_df, fit_wls, term, EXPO, COV_TERMS가 있어야 함
# ==============================================================================
import pandas as pd
from IPython.display import display, HTML

# 1) 모형 적합
sub_df_s3 = sub_df.dropna(subset=['c_pa']).copy()          # 신체활동 비결측자 (S3 동일 표본)
base = f"bmi ~ {EXPO} + {COV_TERMS}"

res_main    = fit_wls(base, sub_df)                         # 주 분석 (전체 완전사례)
res_main_s3 = fit_wls(base, sub_df_s3)                      # S3 동일 표본 기준 (신체활동 미투입)
res_s3      = fit_wls(base + " + C(c_pa)", sub_df_s3)       # S3 (신체활동 추가)

sub_df['exp_rev'] = 8 - sub_df['exp_raw']                   # 클수록 외식이 잦도록 뒤집기
res_s4 = fit_wls(f"bmi ~ exp_rev + {COV_TERMS}", sub_df)    # S4 (연속형)

# 2) 계수 문자열 만들기
def stat(res, name):
    b, se, p = res.params[name], res.bse[name], res.pvalues[name]
    lo, hi = res.conf_int().loc[name]
    ps = f"p = {p:.3f}" if p >= 0.001 else "p < 0.001"
    return b, f"β = {b:.3f} (SE = {se:.3f})<br>95% CI: [{lo:.3f}, {hi:.3f}]<br>{ps}"

def delta(b, ref):
    d = b - ref
    pct = d / abs(ref) * 100
    return f"Δβ = {d:+.4f} ({pct:+.1f}%)"

tm, th = term(res_main, 'mid'), term(res_main, 'high')

_,       main_mid  = stat(res_main, tm)
_,       main_high = stat(res_main, th)
b_ref_m, ref_mid   = stat(res_main_s3, tm)
b_ref_h, ref_high  = stat(res_main_s3, th)
b_s3_m,  s3_mid    = stat(res_s3, tm)
b_s3_h,  s3_high   = stat(res_s3, th)
_,       s4_txt    = stat(res_s4, 'exp_rev')

n_main, n_s3 = len(sub_df), len(sub_df_s3)

# 3) 표 구성
rows = [
    ["주 분석 (모형 2)", f"{n_main:,}명", main_mid, main_high, "기준"],
    ["주 분석 (S3 동일 표본 비교용)", f"{n_s3:,}명", ref_mid, ref_high, "S3 기준 (Ref.)"],
    ["S3. 신체활동 추가 보정", f"{n_s3:,}명", s3_mid, s3_high,
     f"중빈도: {delta(b_s3_m, b_ref_m)}<br>고빈도: {delta(b_s3_h, b_ref_h)}"],
    ["S4. 노출을 연속형으로 (외식 빈도 1단계 증가당)", f"{n_main:,}명", s4_txt,
     "단일 선형 기울기 (중빈도·고빈도 구분 없음)",
     "척도가 달라 Δβ 비교 불가. 방향(+)과 유의성(p<0.05)은 주 분석과 일치"],
]
table3 = pd.DataFrame(rows, columns=[
    "분석 구분", "N", "중빈도 (주 3–4회)", "고빈도 (주 5회 이상)", "주 분석(동일 표본)과의 차이"])

# 4) 표 출력
print("=== [D-2 Table 3] 주 분석 및 민감도 분석 비교 ===")
html_t3 = table3.to_html(index=False, escape=False)
display(HTML(html_t3))

# 5) 점검 (문서에는 옮기지 않음)
print("\n[점검] S4 계수 부호(양수여야 함):", round(res_s4.params['exp_rev'], 3))
print("[점검] S3 N:", n_s3, "| 신체활동 결측으로 제외:", n_main - n_s3)

=== [D-2 Table 3] 주 분석 및 민감도 분석 비교 ===


분석 구분,N,중빈도 (주 3–4회),고빈도 (주 5회 이상),주 분석(동일 표본)과의 차이
주 분석 (모형 2),"5,602명","β = 0.194 (SE = 0.197)95% CI: [-0.193, 0.581]p = 0.326","β = 0.474 (SE = 0.153)95% CI: [0.175, 0.774]p = 0.002",기준
주 분석 (S3 동일 표본 비교용),"5,287명","β = 0.220 (SE = 0.196)95% CI: [-0.164, 0.605]p = 0.261","β = 0.464 (SE = 0.153)95% CI: [0.165, 0.763]p = 0.002",S3 기준 (Ref.)
S3. 신체활동 추가 보정,"5,287명","β = 0.220 (SE = 0.196)95% CI: [-0.164, 0.605]p = 0.262","β = 0.464 (SE = 0.153)95% CI: [0.164, 0.763]p = 0.002",중빈도: Δβ = -0.0003 (-0.1%)고빈도: Δβ = -0.0001 (-0.0%)
S4. 노출을 연속형으로 (외식 빈도 1단계 증가당),"5,602명","β = 0.149 (SE = 0.046)95% CI: [0.060, 0.239]p = 0.001",단일 선형 기울기 (중빈도·고빈도 구분 없음),척도가 달라 Δβ 비교 불가. 방향(+)과 유의성(p<0.05)은 주 분석과 일치



[점검] S4 계수 부호(양수여야 함): 0.149
[점검] S3 N: 5287 | 신체활동 결측으로 제외: 315
